[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jeevash23/logogram/blob/main/notebooks/sae-features.ipynb)

# Features of an SAE on GPT-2 small: read them on a prompt, then patch one

In [ ]:
%pip install logogram

A sparse autoencoder (SAE) rewrites one of the model's activations as a few active features out
of thousands, each a direction in the model, plus an error it misses (Bricken et al., 2023,
"Towards Monosemanticity"; Cunningham et al., 2023, "Sparse Autoencoders Find Highly
Interpretable Features in Language Models"). This notebook loads a published SAE for GPT-2
small's residual stream entering layer 8, reads the features that fire on an IOI prompt, finds
the features that matter for the answer, and patches one of them, as Marks et al. (2024,
"Sparse Feature Circuits") patch features.

The SAE (about 150 MB, from Hugging Face) is read from its safetensors file only.

## Open the example, load the model and the SAE

In [ ]:
import logogram as lg

project = lg.open_example("ioi-gpt2")
model = lg.load_model("openai-community/gpt2")
SAE = {"repo": "jbloom/GPT2-Small-SAEs-Reformatted", "path": "blocks.8.hook_resid_pre"}
sae = lg.load_sae(SAE["repo"], SAE["path"], model=model)

## The features on a prompt

`look` reads any text without making a dataset. With an SAE, it also lists the features that
fire most on each token. SAEs aren't trained on the beginning-of-sequence token, so the listing
starts after it.

In [ ]:
prompt = "When Mary and John went to the store, John gave a drink to"
seen = lg.look(model, prompt, sae=sae, top_k=3)
for i in range(seen["first_real_token"], len(seen["tokens"])):
    strongest = ", ".join(f"{f['feature']} ({f['activation']:.1f})" for f in seen["features"][i])
    print(f"{seen['tokens'][i]!r:>10}  {seen['active'][i]:>3} active · {strongest}")

## What a feature writes

A feature's decoder direction, read straight into the logits through the final normalization and
the unembedding, says which tokens it promotes and suppresses directly. This is the feature that
fires most on the last token.

In [ ]:
feature = seen["features"][-1][0]["feature"]
writes = lg.feature_writes(model, sae, feature, prompt, prepend_bos=True, top=8)
print(f"Feature {feature}, at {writes['token']!r} (activation {writes['activation']:.1f})")
print("promotes:  ", " ".join(repr(t["token"]) for t in writes["promoted"]))
print("suppresses:", " ".join(repr(t["token"]) for t in writes["suppressed"]))

## Which features carry the answer?

Firing strongly isn't the same as mattering. Attribution patching estimates the effect of
patching each of the SAE's 24,576 features at the last token from one gradient, and keeps the 20
strongest. Choosing the strongest of thousands and reporting them on the same prompts biases
their effects away from zero (the winner's curse), so half the prompts, drawn with a seed,
choose the features and the other half report them.

In [ ]:
example = lg.load_spec(project.root / "experiments/ioi-head-patching/spec.json")
features = lg.load_spec(
    {
        **example.model_dump(),
        "name": "Layer 8's SAE features at the last token, estimated",
        "experiment": {
            "kind": "attribution_patching",
            "direction": "clean_to_corrupt",
            "method": "gradient",
            "steps": None,
        },
        "scope": {"kind": "features", "position": {"kind": "last"}, "top": 20, "choose_on": 0.5, "seed": 0},
        "sae": {**SAE, "revision": None},
    }
)
estimate = lg.run(features, project, model=model, sae=sae)
found = estimate.summary["features"]
print(f"The SAE explains {found['fit']['variance_explained']:.0%} of the variance on these prompts,")
print(f"with {found['fit']['l0']:.0f} features active per token.")
print(f"Estimated effect of the whole site {found['site_estimate']:+.3f};")
print(f"its features carry {found['features_estimate']:+.3f} of it, the SAE's error the rest.\n")
for site in estimate.strongest(5):
    print(f"{site['label']:<16}{site['effect']['mean']:>+8.3f}")

At this site the SAE explains most of the activations' variance, yet its features carry only
about a fifth of the site's estimated effect on the answer: the rest flows through its error, the
part of each activation it doesn't reconstruct. The strongest single feature carries a few
percent of the gap. Firing strongly, explaining the variance and mattering for the task are three
different things, and the estimates are first order: patching says what is true.

## Patch one feature

Patching a feature encodes the corrupt prompt's activation, changes that one feature to its
value on the clean prompt, and keeps the SAE's error, so only the feature's direction moves. A
spec names it as a site of kind `sae_feature`.

In [ ]:
strongest = estimate.strongest(1)[0]
one = lg.load_spec(
    {
        **features.model_dump(),
        "name": f"Feature {strongest['feature']} at the last token, patched",
        "experiment": {"kind": "activation_patching", "direction": "clean_to_corrupt"},
        "scope": {
            "kind": "sites",
            "sites": [
                {"kind": "sae_feature", "layer": 8, "feature": strongest["feature"], "position": {"kind": "last"}}
            ],
        },
    }
)
patched = lg.run(one, project, model=model, sae=sae)
site = patched.strongest(1)[0]
e = site["effect"]
print(f"Feature {strongest['feature']}: estimated {strongest['effect']['mean']:+.3f}")
print(f"patched {e['mean']:+.3f}, 95% interval [{e['lo']:+.3f}, {e['hi']:+.3f}], over {site['n']} prompts")
print(f"{site['sign_flips']} of them flip to the clean answer")

`verify_run(estimate, top=10)` patches the ten strongest features the same way. In the app,
**Explore → Features** lists the features on the dataset's prompts and **Your text** on any text,
and **Steer along this feature** adds a feature to the model while it writes.